# Fraud Detection: CatBoost vs. XGBoost

This notebook compares CatBoost and XGBoost with stratified cross-validation for imbalanced fraud detection. It selects the model by training-only average precision, chooses its decision threshold on a separate validation split, and reports performance once on the held-out test set.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from scipy.stats import randint, uniform
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    matthews_corrcoef,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split

RANDOM_STATE = 42


## 1. Load and inspect the data

In [48]:
# Load the feature-engineered transaction dataset.
data_path = Path("../Datasets/Transaction_risk_insights_feature_engineered.csv")
if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found: {data_path.resolve()}")

df = pd.read_csv(data_path)
print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()


Dataset shape: 550,000 rows x 30 columns


,transaction_id,customer_id,transaction_date,transaction_time,account_type,transaction_type,transaction_amount,transaction_direction,account_balance,merchant_category,...,transaction_year,transaction_month,transaction_day,day_of_week,is_weekend,time_period,transaction_amount_category,balance_category,credit_score_category,customer_transaction_count
0,TXN000000001,CUST015796,2019-01-01,1900-01-01 15:28:00,Current,UPI,1820.17,Debit,609365.31,Food & Dining,...,2019,1,1,Tuesday,0,Afternoon,Low,High,High,10
1,TXN000000002,CUST000861,2019-01-01,1900-01-01 03:00:00,Current,UPI,392.67,Credit,14451.14,Education,...,2019,1,1,Tuesday,0,Night,Low,Low,Medium,8
2,TXN000000003,CUST076821,2019-01-01,1900-01-01 18:03:00,Fixed Deposit,POS,1255.97,Debit,47621.87,Utilities,...,2019,1,1,Tuesday,0,Evening,Low,Medium,High,5
3,TXN000000004,CUST054887,2019-01-01,1900-01-01 08:03:00,Savings,UPI,2580.68,Debit,34467.85,Travel,...,2019,1,1,Tuesday,0,Morning,Low,Medium,Medium,7
4,TXN000000005,CUST006266,2019-01-01,1900-01-01 14:23:00,Fixed Deposit,UPI,2573.80,Debit,26617.40,Utilities,...,2019,1,1,Tuesday,0,Afternoon,Low,Medium,High,10


## 2. Select and prepare model features

In [49]:
# Use transaction-time attributes only; exclude identifiers and the target.
features = [
    "account_type",
    "transaction_type",
    "transaction_amount",
    "transaction_direction",
    "account_balance",
    "merchant_category",
    "state",
    "credit_score",
    "has_loan",
    "loan_type",
    "emi_amount",
    "channel",
    "kyc_status",
    "transaction_hour",
    "transaction_month",
    "day_of_week",
    "is_weekend",
]


In [50]:
target = "is_fraud"
missing_columns = sorted(set(features + [target]) - set(df.columns))
if missing_columns:
    raise KeyError(f"Missing required dataset columns: {missing_columns}")

X = df.loc[:, features].copy()
y = df[target].astype("int8").copy()
print(f"Feature matrix: {X.shape}; target distribution: {y.value_counts().sort_index().to_dict()}")


Feature matrix: (550000, 17); target distribution: {0: 545127, 1: 4873}


In [51]:
# Log-transform skewed non-negative monetary features.
for column in ("transaction_amount", "account_balance", "emi_amount"):
    if (X[column] < 0).any():
        raise ValueError(f"{column} contains negative values; log1p is not valid.")
    X[f"{column}_log"] = np.log1p(X[column])

X = X.drop(columns=["transaction_amount", "account_balance", "emi_amount"])
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
print(f"Prepared features: {X.shape[1]} total; {len(categorical_cols)} categorical")
print("Categorical columns:", categorical_cols)


Prepared features: 17 total; 9 categorical
Categorical columns: ['account_type', 'transaction_type', 'transaction_direction', 'merchant_category', 'state', 'loan_type', 'channel', 'kyc_status', 'day_of_week']


## 3. Create stratified train, validation, and test splits

In [52]:
# Reserve a stratified test set, then split training data for tuning and threshold selection.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)
X_tune_train, X_valid, y_tune_train, y_valid = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_train,
)

print(f"Tuning/training rows: {len(X_tune_train):,}")
print(f"Threshold-validation rows: {len(X_valid):,}")
print(f"Held-out test rows: {len(X_test):,}")


Tuning/training rows: 352,000
Threshold-validation rows: 88,000
Held-out test rows: 110,000


In [53]:
# Compute imbalance weights from the model-training partition only.
negative_count = int((y_tune_train == 0).sum())
positive_count = int((y_tune_train == 1).sum())
if positive_count == 0:
    raise ValueError("The model-training partition contains no fraud examples.")

# Python floats keep estimator parameters compatible with cloning and serialization.
class_weights = (1.0, float(negative_count / positive_count))
print(f"Non-fraud: {negative_count:,}; fraud: {positive_count:,}")
print(f"Class weights: {class_weights}")


Non-fraud: 348,882; fraud: 3,118
Class weights: (1.0, 111.89288005131495)


## 4. Tune CatBoost and compare XGBoost

In [54]:
# Search a compact range suited to imbalanced tabular classification.
catboost_param_distributions = {
    "iterations": randint(150, 401),
    "learning_rate": uniform(0.02, 0.13),
    "depth": randint(4, 9),
    "l2_leaf_reg": uniform(2.0, 8.0),
    "random_strength": uniform(0.1, 2.9),
    "bagging_temperature": uniform(0.0, 3.0),
}


In [55]:
catboost_search = RandomizedSearchCV(
    estimator=CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="PRAUC",
        class_weights=class_weights,
        random_seed=RANDOM_STATE,
        verbose=False,
        thread_count=1,
        allow_writing_files=False,
    ),
    param_distributions=catboost_param_distributions,
    n_iter=6,
    scoring="average_precision",
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    refit=True,
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbose=1,
)


In [56]:
# Tune against the same average-precision metric used for model evaluation.
# Use a stratified sample for search, then refit the winner on all tuning rows.
search_size = min(80_000, len(X_tune_train))
if search_size < len(X_tune_train):
    X_search, _, y_search, _ = train_test_split(
        X_tune_train,
        y_tune_train,
        train_size=search_size,
        random_state=RANDOM_STATE,
        stratify=y_tune_train,
    )
else:
    X_search = X_tune_train
    y_search = y_tune_train
catboost_search.fit(
    X_search,
    y_search,
    cat_features=categorical_cols,
)
best_params = catboost_search.best_params_
print(f"Best 3-fold CV average precision: {catboost_search.best_score_:.4f}")
print("Best CatBoost parameters:", best_params)

catboost_model = CatBoostClassifier(
    **best_params,
    loss_function="Logloss",
    eval_metric="PRAUC",
    class_weights=class_weights,
    random_seed=RANDOM_STATE,
    verbose=100,
    thread_count=-1,
    allow_writing_files=False,
)
catboost_model.fit(
    X_tune_train,
    y_tune_train,
    cat_features=categorical_cols,
)


Fitting 3 folds for each of 6 candidates, totalling 18 fits
Best 3-fold CV average precision: 0.0196
Best CatBoost parameters: {'bagging_temperature': np.float64(1.1236203565420875), 'depth': 8, 'iterations': 164, 'l2_leaf_reg': np.float64(7.855951534491241), 'learning_rate': np.float64(0.09782560294561476), 'random_strength': np.float64(0.5524540572830658)}
0:	learn: 0.6090646	total: 110ms	remaining: 17.9s
100:	learn: 0.8101490	total: 9.14s	remaining: 5.7s
163:	learn: 0.8743193	total: 16.2s	remaining: 0us


CatBoostClassifier(allow_writing_files=False, bagging_temperature=np.float64(1.1236203565420875), class_weights=(1.0, 111.89288005131495), depth=8, eval_metric='PRAUC', iterations=164, l2_leaf_reg=np.float64(7.855951534491241), learning_rate=np.float64(0.09782560294561476), loss_function='Logloss', random_seed=42, random_strength=np.float64(0.5524540572830658), verbose=100)

In [ ]:
# Compare XGBoost with the tuned CatBoost baseline using the same training-only CV folds.
# Categories are encoded natively and unknown values get an explicit fallback level.
xgb_category_levels = {}
for column in categorical_cols:
    levels = sorted(X_tune_train[column].fillna("__missing__").astype(str).unique().tolist())
    if "__unknown__" not in levels:
        levels.append("__unknown__")
    xgb_category_levels[column] = levels


def xgb_frame(frame):
    prepared = frame.copy()
    for column, levels in xgb_category_levels.items():
        values = prepared[column].fillna("__missing__").astype(str)
        values = values.where(values.isin(levels), "__unknown__")
        prepared[column] = pd.Categorical(values, categories=levels)
    return prepared


xgb_imbalance = negative_count / positive_count
xgb_candidates = [
    {"max_depth": 3, "min_child_weight": 5, "scale_pos_weight": 1.0},
    {"max_depth": 4, "min_child_weight": 5, "scale_pos_weight": float(np.sqrt(xgb_imbalance))},
]
xgb_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
best_xgb_score = -np.inf
best_xgb_params = None
for candidate in xgb_candidates:
    fold_scores = []
    for fold, (train_idx, valid_idx) in enumerate(xgb_cv.split(X_search, y_search), start=1):
        candidate_model = XGBClassifier(
            **candidate,
            n_estimators=120,
            learning_rate=0.04,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=2.0,
            objective="binary:logistic",
            eval_metric="aucpr",
            tree_method="hist",
            enable_categorical=True,
            random_state=RANDOM_STATE + fold,
            n_jobs=-1,
        )
        candidate_model.fit(
            xgb_frame(X_search.iloc[train_idx]), y_search.iloc[train_idx], verbose=False,
        )
        fold_probability = candidate_model.predict_proba(xgb_frame(X_search.iloc[valid_idx]))[:, 1]
        fold_scores.append(average_precision_score(y_search.iloc[valid_idx], fold_probability))

    candidate_score = float(np.mean(fold_scores))
    print(f"XGBoost {candidate}: 3-fold CV AP={candidate_score:.4f}")
    if candidate_score > best_xgb_score:
        best_xgb_score = candidate_score
        best_xgb_params = dict(candidate)

print(f"CatBoost best CV AP: {catboost_search.best_score_:.4f}")
print(f"XGBoost best CV AP:  {best_xgb_score:.4f}")
if best_xgb_score > catboost_search.best_score_:
    selected_algorithm = "XGBoost"
    selected_model = XGBClassifier(
        **best_xgb_params,
        n_estimators=120,
        learning_rate=0.04,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=2.0,
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        enable_categorical=True,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    selected_model.fit(xgb_frame(X_tune_train), y_tune_train, verbose=False)
else:
    selected_algorithm = "CatBoost"
    selected_model = catboost_model
print(f"Selected model by CV average precision: {selected_algorithm}")


## 5. Select a threshold and evaluate the final model

In [ ]:
# Select the operating threshold on validation data, never on the held-out test set.
valid_features = xgb_frame(X_valid) if selected_algorithm == "XGBoost" else X_valid
valid_probabilities = selected_model.predict_proba(valid_features)[:, 1]
valid_precision, valid_recall, thresholds = precision_recall_curve(
    y_valid,
    valid_probabilities,
)
valid_f1 = (
    2 * valid_precision[:-1] * valid_recall[:-1]
    / (valid_precision[:-1] + valid_recall[:-1] + 1e-12)
)
best_threshold_index = int(np.argmax(valid_f1))
decision_threshold = float(thresholds[best_threshold_index])
print(f"Validation-selected threshold (max F1): {decision_threshold:.4f}")
print(f"Validation F1 at selected threshold: {valid_f1[best_threshold_index]:.4f}")

# Evaluate the selected model and threshold once on the untouched test set.
test_features = xgb_frame(X_test) if selected_algorithm == "XGBoost" else X_test
test_probabilities = selected_model.predict_proba(test_features)[:, 1]
test_predictions = (test_probabilities >= decision_threshold).astype("int8")
test_average_precision = average_precision_score(y_test, test_probabilities)
test_baseline_ap = float(y_test.mean())
print(f"\nTest average precision: {test_average_precision:.4f}")
print(f"Test ROC-AUC: {roc_auc_score(y_test, test_probabilities):.4f}")
print(f"No-skill baseline (fraud prevalence): {test_baseline_ap:.4f}")
print(f"Average-precision lift over baseline: {test_average_precision / test_baseline_ap:.2f}x")
print(f"Test MCC at selected threshold: {matthews_corrcoef(y_test, test_predictions):.4f}")
print("\nTest classification report:")
print(
    classification_report(
        y_test,
        test_predictions,
        labels=[0, 1],
        target_names=["Non-Fraud", "Fraud"],
        zero_division=0,
    )
)

matrix = confusion_matrix(y_test, test_predictions, labels=[0, 1])
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(
    matrix,
    annot=True,
    fmt=",d",
    cmap="Blues",
    xticklabels=["Non-Fraud", "Fraud"],
    yticklabels=["Non-Fraud", "Fraud"],
    ax=ax,
)
ax.set(title=f"{selected_algorithm} - Held-out Test Confusion Matrix", xlabel="Predicted", ylabel="Actual")
fig.tight_layout()
plt.show()
